# Urban Analytics — Los Angeles
Exploração do dataset integrado `urban_dataset_2025.csv`

**Colunas:** `timestamp · crime_count · traffic_flow · avg_speed · temperature · precipitation`

- Crime: 2020–2025
- Tráfego + Clima: 2025 (overlap completo no ano de 2025)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 110

## 1. Carga e visão geral

In [ ]:
df = pd.read_csv('../data/processed/urban_dataset_2025.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Linhas: {len(df):,}  |  Período: {df.timestamp.min().date()} → {df.timestamp.max().date()}')
df.info()

In [ ]:
df.describe()

In [ ]:
nulls = df.isnull().sum()
pct   = (nulls / len(df) * 100).round(1)
pd.DataFrame({'nulos': nulls, '%': pct})

---
## 2. Crime — série histórica (2020–2025)

In [ ]:
crime_daily = df.set_index('timestamp')['crime_count'].resample('D').sum()
crime_daily_smooth = crime_daily.rolling(30, center=True).mean()

fig, ax = plt.subplots()
ax.bar(crime_daily.index, crime_daily.values, color='steelblue', alpha=0.35, label='Diário')
ax.plot(crime_daily_smooth.index, crime_daily_smooth.values, color='crimson', lw=2, label='Média móvel 30d')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_title('Crimes diários — Los Angeles (2020–2025)')
ax.set_ylabel('Nº de crimes')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
crime_monthly = df.set_index('timestamp')['crime_count'].resample('ME').sum().reset_index()

fig, ax = plt.subplots()
ax.plot(crime_monthly['timestamp'], crime_monthly['crime_count'], marker='o', ms=4, color='steelblue')
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
ax.set_title('Crimes mensais — Los Angeles')
ax.set_ylabel('Total de crimes')
plt.tight_layout()
plt.show()

In [ ]:
df['hour'] = df['timestamp'].dt.hour
crime_by_hour = df.groupby('hour')['crime_count'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(crime_by_hour.index, crime_by_hour.values, color='steelblue', edgecolor='white')
# Destaca hora de pico
peak_hour = crime_by_hour.idxmax()
bars[peak_hour].set_color('crimson')
ax.axvline(peak_hour, color='crimson', ls='--', alpha=0.5, label=f'Pico: {peak_hour}h')
ax.set_xticks(range(0, 24))
ax.set_title('Média de crimes por hora do dia (2020–2025)')
ax.set_xlabel('Hora')
ax.set_ylabel('Média de crimes')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Tráfego — 2025

In [ ]:
df25 = df[df['timestamp'].dt.year == 2025].copy()
print(f'Linhas 2025: {len(df25):,}  |  Nulos traffic_flow: {df25.traffic_flow.isnull().sum():,}')

In [ ]:
t_daily = df25.set_index('timestamp')[['traffic_flow', 'avg_speed']].resample('D').agg(
    {'traffic_flow': 'sum', 'avg_speed': 'mean'}
).dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].fill_between(t_daily.index, t_daily['traffic_flow'], alpha=0.6, color='teal')
axes[0].set_ylabel('Fluxo total (veículos)')
axes[0].set_title('Tráfego diário — Los Angeles 2025')

axes[1].plot(t_daily.index, t_daily['avg_speed'], color='darkorange', lw=1.5)
axes[1].set_ylabel('Velocidade média (mph)')
axes[1].set_xlabel('Data')

plt.tight_layout()
plt.show()

In [ ]:
traffic_by_hour = df25.groupby('hour')[['traffic_flow', 'avg_speed']].mean()

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()

ax1.bar(traffic_by_hour.index, traffic_by_hour['traffic_flow'], color='teal', alpha=0.5, label='Fluxo')
ax2.plot(traffic_by_hour.index, traffic_by_hour['avg_speed'], color='darkorange', lw=2, marker='o', ms=4, label='Velocidade')

ax1.set_xticks(range(0, 24))
ax1.set_xlabel('Hora')
ax1.set_ylabel('Fluxo médio (veículos)', color='teal')
ax2.set_ylabel('Velocidade média (mph)', color='darkorange')
ax1.set_title('Perfil horário de tráfego — 2025')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

---
## 4. Clima — 2025

In [ ]:
w_daily = df25.set_index('timestamp')[['temperature', 'precipitation']].resample('D').agg(
    {'temperature': 'mean', 'precipitation': 'sum'}
).dropna(subset=['temperature'])

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(w_daily.index, w_daily['temperature'], color='tomato', lw=1.5)
axes[0].fill_between(w_daily.index, w_daily['temperature'], alpha=0.2, color='tomato')
axes[0].set_ylabel('Temperatura (°C)')
axes[0].set_title('Temperatura e Precipitação — Los Angeles 2025')

axes[1].bar(w_daily.index, w_daily['precipitation'], color='royalblue', alpha=0.7)
axes[1].set_ylabel('Precipitação (mm)')
axes[1].set_xlabel('Data')

plt.tight_layout()
plt.show()

---
## 5. Análise integrada — 2025 (crime + tráfego + clima)

In [ ]:
daily = df25.set_index('timestamp').resample('D').agg(
    crime_count=('crime_count', 'sum'),
    traffic_flow=('traffic_flow', 'sum'),
    avg_speed=('avg_speed', 'mean'),
    temperature=('temperature', 'mean'),
    precipitation=('precipitation', 'sum'),
).dropna(subset=['traffic_flow'])

# Normalizar para mesmo eixo
def normalize(s):
    return (s - s.min()) / (s.max() - s.min())

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily.index, normalize(daily['crime_count']), label='Crime (norm)', color='crimson', alpha=0.8)
ax.plot(daily.index, normalize(daily['traffic_flow']), label='Tráfego (norm)', color='teal', alpha=0.8)
ax.plot(daily.index, normalize(daily['temperature']), label='Temperatura (norm)', color='tomato', ls='--', alpha=0.7)
ax.bar(daily.index, normalize(daily['precipitation']), color='royalblue', alpha=0.25, label='Precipitação (norm)', width=1)

ax.set_title('Variáveis normalizadas — Los Angeles 2025')
ax.set_ylabel('Valor normalizado [0–1]')
ax.legend(ncol=4)
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['crime_count', 'traffic_flow', 'avg_speed', 'temperature', 'precipitation']
corr = daily[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax
)
ax.set_title('Correlação entre variáveis urbanas — 2025')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

pairs = [
    ('traffic_flow', 'crime_count',   'Tráfego vs Crime',       'teal',      'crimson'),
    ('temperature',  'traffic_flow',  'Temperatura vs Tráfego', 'tomato',    'teal'),
    ('precipitation','avg_speed',     'Chuva vs Velocidade',    'royalblue', 'darkorange'),
]

for ax, (x, y, title, cx, cy) in zip(axes, pairs):
    data = daily[[x, y]].dropna()
    ax.scatter(data[x], data[y], alpha=0.4, color=cx, s=20)
    # linha de tendência
    z = np.polyfit(data[x], data[y], 1)
    p = np.poly1d(z)
    xs = np.linspace(data[x].min(), data[x].max(), 200)
    ax.plot(xs, p(xs), color=cy, lw=2)
    ax.set_xlabel(x.replace('_', ' '))
    ax.set_ylabel(y.replace('_', ' '))
    ax.set_title(title)

plt.tight_layout()
plt.show()

---
## 6. Heatmap — dia da semana × hora do dia

In [ ]:
df25['dow'] = df25['timestamp'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

pivot_crime = df25.pivot_table(
    values='crime_count', index='dow', columns='hour', aggfunc='mean'
).reindex(dow_order)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot_crime, cmap='YlOrRd', linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Média de crimes'})
ax.set_title('Crimes — dia da semana × hora (2025)')
ax.set_xlabel('Hora do dia')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
pivot_flow = df25.pivot_table(
    values='traffic_flow', index='dow', columns='hour', aggfunc='mean'
).reindex(dow_order)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot_flow, cmap='Blues', linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Fluxo médio'})
ax.set_title('Fluxo de tráfego — dia da semana × hora (2025)')
ax.set_xlabel('Hora do dia')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

---
## 7. Resumo de insights

Execute a célula abaixo para gerar um resumo automático com os principais valores observados.

In [ ]:
peak_crime_hour  = df['groupby'  if False else 'hour'].pipe(lambda _: df.groupby('hour')['crime_count'].mean().idxmax())
peak_traffic_hour = df25.groupby('hour')['traffic_flow'].mean().idxmax()
slowest_hour      = df25.groupby('hour')['avg_speed'].mean().idxmin()
corr_rain_speed   = daily[['precipitation','avg_speed']].corr().iloc[0,1]
corr_traffic_crime= daily[['traffic_flow','crime_count']].corr().iloc[0,1]

print('=' * 55)
print('  INSIGHTS — Urban Analytics Los Angeles')
print('=' * 55)
print(f'  Hora pico de crimes  : {peak_crime_hour}h')
print(f'  Hora pico de tráfego : {peak_traffic_hour}h')
print(f'  Hora mais lenta      : {slowest_hour}h (menor velocidade média)')
print(f'  Corr chuva↔velocidade: {corr_rain_speed:.3f}')
print(f'  Corr tráfego↔crime   : {corr_traffic_crime:.3f}')
print('=' * 55)